# RC0 VAE-latent carrier diagnostic execution shell

This notebook is a thin RC0-DIAG-CARRIER-SWITCH-VAE-LATENT execution shell only. It contains no generation hook, decoder, evaluator, scientific formula, threshold, externally supplied run identity, data, credential, or Drive identifier. It constructs a `DIAGNOSTIC_ONLY` identity from an explicit exact clean Git ref.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import hashlib, json, os, re, shutil, stat, subprocess, sys, zipfile

REPOSITORY_URL = ''
AUTHORIZED_REF = ''  # frozen diagnostic implementation
LOCAL_OUTPUT_ROOT = '/content/rc0-output'
DRIVE_OUTPUT_ROOT = '/content/drive/MyDrive/SC-SSTW-Feasibility/rc0-diag-vae-latent-inference-mode'
AUTHORIZE_EXECUTION = False
AUTHORIZE_DRIVE_IO = False
if not AUTHORIZE_EXECUTION or not REPOSITORY_URL or re.fullmatch(r'[0-9a-f]{40}', AUTHORIZED_REF) is None:
    raise RuntimeError('explicit exact diagnostic ref, repository, and user execution acknowledgement are required')


In [ ]:
def _exists_lstat(path):
    try:
        path.lstat()
        return True
    except FileNotFoundError:
        return False

def _require_absent(paths):
    for path in paths:
        if _exists_lstat(path):
            raise RuntimeError(f'run target preexisting: {path}')

def _require_real_directory(path, expected):
    metadata = path.lstat()
    if stat.S_ISLNK(metadata.st_mode) or not stat.S_ISDIR(metadata.st_mode):
        raise RuntimeError(f'unsafe directory target: {path}')
    if path.resolve(strict=True) != expected.resolve(strict=True):
        raise RuntimeError(f'directory realpath mismatch: {path}')

def _write_bytes_exclusive(path, payload):
    with path.open('xb') as handle:
        handle.write(payload)
        handle.flush()
        os.fsync(handle.fileno())

def _copy_file_exclusive(source, destination):
    with source.open('rb') as read_handle, destination.open('xb') as write_handle:
        shutil.copyfileobj(read_handle, write_handle)
        write_handle.flush()
        os.fsync(write_handle.fileno())

def _copy_tree_exclusive(source_root, destination_root):
    source_metadata = source_root.lstat()
    if stat.S_ISLNK(source_metadata.st_mode) or not stat.S_ISDIR(source_metadata.st_mode):
        raise RuntimeError(f'copy source is not a real directory: {source_root}')
    destination_root.mkdir(mode=0o700, parents=False, exist_ok=False)
    for current_text, directory_names, file_names in os.walk(source_root, topdown=True, followlinks=False):
        current = Path(current_text)
        relative = current.relative_to(source_root)
        destination_current = destination_root / relative
        for name in sorted(directory_names):
            source = current / name
            metadata = source.lstat()
            if stat.S_ISLNK(metadata.st_mode) or not stat.S_ISDIR(metadata.st_mode):
                raise RuntimeError(f'copy source directory unsafe: {source}')
            (destination_current/name).mkdir(mode=0o700, parents=False, exist_ok=False)
        for name in sorted(file_names):
            source = current / name
            metadata = source.lstat()
            if stat.S_ISLNK(metadata.st_mode) or not stat.S_ISREG(metadata.st_mode):
                raise RuntimeError(f'copy source file unsafe: {source}')
            _copy_file_exclusive(source, destination_current/name)

def _zip_tree_exclusive(source_root, archive):
    with zipfile.ZipFile(archive, mode='x', compression=zipfile.ZIP_DEFLATED) as handle:
        for source in sorted(source_root.rglob('*')):
            metadata = source.lstat()
            if stat.S_ISLNK(metadata.st_mode):
                raise RuntimeError(f'archive source symlink forbidden: {source}')
            if stat.S_ISREG(metadata.st_mode):
                handle.write(source, source.relative_to(source_root).as_posix())

def _validated_actual_package(raw_path, output):
    candidate = Path(raw_path)
    if not candidate.is_absolute() or candidate.parent != output.parent:
        raise RuntimeError('runner package path escaped the frozen output parent')
    if candidate.name != output.name and not candidate.name.startswith(output.name + '.invalid.'):
        raise RuntimeError('runner package name is outside the frozen output family')
    parent_real = output.parent.resolve(strict=True)
    if candidate.parent.resolve(strict=True) != parent_real:
        raise RuntimeError('runner package parent realpath mismatch')
    metadata = candidate.lstat()
    if stat.S_ISLNK(metadata.st_mode) or not stat.S_ISDIR(metadata.st_mode):
        raise RuntimeError('runner package is not a real directory')
    return candidate

RUN_ID = None
diagnostic_bootstrap_sha = None
DIAGNOSTIC_BOOTSTRAP = None
WORK = LOG = OUTPUT = ARCHIVE_ROOT = ARCHIVE = SIDECAR = None
DRIVE_ARCHIVE = DRIVE_SIDECAR = None
actual_package_path = None
log_owned = False
runner_invocation_attempted = False
runner_started = False
audit = None
completed = None
caught = None
archive_error = None
archive_sha = None
state = {
    'schema': 'sc-sstw.rc0.notebook-execution-state.v1',
    'authorized_ref': AUTHORIZED_REF,
    'diagnostic_class': 'DIAGNOSTIC_ONLY',
    'diagnostic_bootstrap_sha256': None,
    'run_id': None,
    'runner_invocation_attempted': False,
    'runner_started': False,
    'runner_returned': False,
    'audit_loaded': False,
    'audit_status': None,
    'actual_package_path': None,
    'failure_type': None,
}
try:
    RUN_ID = hashlib.sha256(('RC0-DIAG-VAE-LATENT-INFERENCE-MODE:' + AUTHORIZED_REF).encode()).hexdigest()[:16]
    state.update({'run_id': RUN_ID})
    WORK = Path('/content') / f'sc-sstw-rc0-clean-{RUN_ID}'
    LOG = Path('/content') / f'rc0-notebook-log-{RUN_ID}'
    output_root = Path(LOCAL_OUTPUT_ROOT)
    OUTPUT = output_root / RUN_ID
    ARCHIVE_ROOT = Path('/content') / f'rc0-{RUN_ID}-failure-safe'
    ARCHIVE = Path('/content') / f'rc0-{RUN_ID}.zip'
    SIDECAR = Path('/content') / f'rc0-{RUN_ID}.zip.sha256.json'
    _require_absent([WORK, LOG, OUTPUT, ARCHIVE_ROOT, ARCHIVE, SIDECAR])
    expected_output_root = Path('/content/rc0-output')
    if output_root != expected_output_root:
        raise RuntimeError('LOCAL_OUTPUT_ROOT must remain the frozen local root')
    try:
        output_root.mkdir(mode=0o700, parents=False, exist_ok=False)
    except FileExistsError:
        pass
    _require_real_directory(output_root, expected_output_root)
    _require_absent([OUTPUT])
    LOG.mkdir(mode=0o700, parents=False, exist_ok=False)
    log_owned = True
    _write_bytes_exclusive(LOG/'execution_state.initial.json', (json.dumps(state, sort_keys=True, indent=2)+'\n').encode())
    with (LOG/'clone.stdout').open('xb') as clone_stdout, (LOG/'clone.stderr').open('xb') as clone_stderr:
        subprocess.run(['git', 'clone', '--no-checkout', REPOSITORY_URL, str(WORK)], check=True, stdout=clone_stdout, stderr=clone_stderr)
    with (LOG/'checkout.stdout').open('xb') as checkout_stdout, (LOG/'checkout.stderr').open('xb') as checkout_stderr:
        subprocess.run(['git', 'checkout', '--detach', AUTHORIZED_REF], cwd=WORK, check=True, stdout=checkout_stdout, stderr=checkout_stderr)
    head = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=WORK, check=True, capture_output=True, text=True).stdout.strip()
    tree = subprocess.run(['git', 'rev-parse', 'HEAD^{tree}'], cwd=WORK, check=True, capture_output=True, text=True).stdout.strip()
    dirty = subprocess.run(['git', 'status', '--porcelain=v1', '--untracked-files=all'], cwd=WORK, check=True, capture_output=True, text=True).stdout
    if head != AUTHORIZED_REF or dirty:
        raise RuntimeError('detached clean exact-ref preflight failed')
    runtime = {'head': head, 'tree': tree, 'python': sys.version, 'cuda_visible_devices': os.environ.get('CUDA_VISIBLE_DEVICES'), 'diagnostic_class': 'DIAGNOSTIC_ONLY'}
    _write_bytes_exclusive(LOG/'runtime.initial.json', (json.dumps(runtime, sort_keys=True, indent=2)+'\n').encode())
    if subprocess.run(['nvidia-smi'], capture_output=True).returncode != 0:
        raise RuntimeError('authorized CUDA runtime is unavailable')
    locked = ['accelerate==1.4.0', 'diffusers==0.35.2', 'ftfy==6.3.1', 'huggingface_hub==0.35.3', 'imageio==2.37.0', 'imageio_ffmpeg==0.6.0', 'numpy==1.26.4', 'safetensors==0.5.3', 'transformers==4.49.0']
    with (LOG/'pip.stdout').open('xb') as pip_stdout, (LOG/'pip.stderr').open('xb') as pip_stderr:
        subprocess.run([sys.executable, '-m', 'pip', 'install', *locked], check=True, stdout=pip_stdout, stderr=pip_stderr)
    from importlib.metadata import version
    observed_versions = {requirement.split('==')[0]: version(requirement.split('==')[0]) for requirement in locked}
    if any(observed_versions[name] != requirement.split('==')[1] for requirement in locked for name in [requirement.split('==')[0]]):
        raise RuntimeError('locked dependency identity check failed')
    import torch
    if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported() or torch.__version__.split('.', 1)[0] != '2':
        raise RuntimeError('torch CUDA/BF16 runtime policy failed')
    runtime.update({'torch': torch.__version__, 'cuda': torch.version.cuda, 'gpu': torch.cuda.get_device_name(0), 'locked_versions': observed_versions})
    _write_bytes_exclusive(LOG/'runtime.final.json', (json.dumps(runtime, sort_keys=True, indent=2)+'\n').encode())
    sys.path.insert(0, str(WORK/'src'))
    from sc_sstw_feasibility.rc0_minimal_implementation import CONFIG_PATH, CONFIG_RAW_SHA256, DIAGNOSTIC_BOOTSTRAP_SCHEMA, DIAGNOSTIC_CLASS, DIAGNOSTIC_CONFIG_PATH, DIAGNOSTIC_PROTOCOL_PATH, EXTRACTOR_IDENTITY, IMPLEMENTATION_SCHEMA, MANIFEST_SCHEMA, NOTEBOOK_PATH, OUTPUT_SCHEMA, PLAN_PATH, PLAN_RAW_SHA256, PROTOCOL_ID, PROTOCOL_PATH, PROTOCOL_RAW_SHA256, REQUIRED_SOURCE_PATHS, RUNNER_PATH
    frozen_config = json.loads((WORK/CONFIG_PATH).read_text())
    frozen_plan = json.loads((WORK/PLAN_PATH).read_text())
    source_files = {relative: hashlib.sha256((WORK/relative).read_bytes()).hexdigest() for relative in REQUIRED_SOURCE_PATHS}
    diagnostic_bootstrap = {
        'schema_version': 1,
        'bootstrap_schema': DIAGNOSTIC_BOOTSTRAP_SCHEMA,
        'diagnostic_class': DIAGNOSTIC_CLASS,
        'implementation_schema': IMPLEMENTATION_SCHEMA,
        'protocol_id': PROTOCOL_ID,
        'expected_source': {'head': head, 'tree': tree, 'dirty': False},
        'source_files': source_files,
        'frozen_identities': {
            'config': {'path': CONFIG_PATH, 'sha256': CONFIG_RAW_SHA256},
            'plan': {'path': PLAN_PATH, 'sha256': PLAN_RAW_SHA256},
            'protocol': {'path': PROTOCOL_PATH, 'sha256': PROTOCOL_RAW_SHA256},
            'notebook': {'path': NOTEBOOK_PATH, 'sha256': source_files[NOTEBOOK_PATH]},
            'extractor': dict(EXTRACTOR_IDENTITY),
        },
        'diagnostic_identities': {
            'protocol': {'path': DIAGNOSTIC_PROTOCOL_PATH, 'sha256': source_files[DIAGNOSTIC_PROTOCOL_PATH]},
            'config': {'path': DIAGNOSTIC_CONFIG_PATH, 'sha256': source_files[DIAGNOSTIC_CONFIG_PATH]},
        },
        'experiment': {
            'group_order': frozen_plan['group_order'],
            'condition_order': frozen_plan['condition_order'],
            'attempt_order': frozen_plan['attempts'],
            'attempt_budget': frozen_plan['attempt_budget'],
            'retry_policy': frozen_plan['retry_policy'],
        },
        'evidence_policy': {
            'diagnostic_only': True,
            'authorization_claimed': False,
            'gate_claimed': False,
            'production_entry': 'same_process_runner_generate_only',
            'external_production_package_permitted': False,
            'formal_result': False,
            'stage_progression_allowed': False,
        },
        'command_schema': {
            'runner': RUNNER_PATH,
            'required_arguments': ['--generate', '--diagnostic-bootstrap', '--output', '--source-commit'],
            'forbidden_arguments': ['--execution-package', '--synthetic-fixture', '--backend', '--device', '--receipt'],
        },
        'output_schema': OUTPUT_SCHEMA,
    }
    diagnostic_bootstrap_bytes = (json.dumps(diagnostic_bootstrap, separators=(',', ':'), allow_nan=False)+'\n').encode()
    diagnostic_bootstrap_sha = hashlib.sha256(diagnostic_bootstrap_bytes).hexdigest()
    DIAGNOSTIC_BOOTSTRAP = LOG/'diagnostic_bootstrap.json'
    _write_bytes_exclusive(DIAGNOSTIC_BOOTSTRAP, diagnostic_bootstrap_bytes)
    state['diagnostic_bootstrap_sha256'] = diagnostic_bootstrap_sha
    model_id = frozen_config['model']['id']
    model_revision = frozen_config['model']['revision']
    if not isinstance(model_id, str) or not model_id or not isinstance(model_revision, str) or re.fullmatch(r'[0-9a-f]{40}', model_revision) is None:
        raise RuntimeError('frozen model id/revision identity is invalid')
    from huggingface_hub import snapshot_download
    snapshot_result = snapshot_download(repo_id=model_id, revision=model_revision, local_files_only=False)
    snapshot_path = Path(snapshot_result)
    snapshot_metadata = snapshot_path.lstat()
    if stat.S_ISLNK(snapshot_metadata.st_mode) or not stat.S_ISDIR(snapshot_metadata.st_mode):
        raise RuntimeError('downloaded model snapshot is not a real directory')
    resolved_snapshot = snapshot_path.resolve(strict=True)
    if snapshot_path.name != model_revision or resolved_snapshot.name != model_revision:
        raise RuntimeError('downloaded model snapshot commit identity mismatch')
    _write_bytes_exclusive(LOG/'model_snapshot.json', (json.dumps({'model_id': model_id, 'revision': model_revision, 'snapshot_path': str(resolved_snapshot)}, sort_keys=True, indent=2)+'\n').encode())
    if AUTHORIZE_DRIVE_IO:
        if not DRIVE_OUTPUT_ROOT:
            raise RuntimeError('explicit Drive destination is required')
        drive_root = Path(DRIVE_OUTPUT_ROOT)
        if not drive_root.is_absolute() or Path('/content/drive') not in drive_root.parents:
            raise RuntimeError('Drive destination must be below the mounted Drive root')
        drive_project_root = Path('/content/drive/MyDrive/SC-SSTW-Feasibility')
        if drive_root.parent != drive_project_root:
            raise RuntimeError('Drive destination differs from the frozen diagnostic root')
        if not _exists_lstat(drive_project_root):
            drive_project_root.mkdir(mode=0o700, parents=False, exist_ok=False)
        _require_real_directory(drive_project_root, drive_project_root)
        if not _exists_lstat(drive_root):
            drive_root.mkdir(mode=0o700, parents=False, exist_ok=False)
        _require_real_directory(drive_root, drive_root)
        if not os.access(drive_root, os.W_OK):
            raise RuntimeError('Drive destination is not writable')
        DRIVE_ARCHIVE = drive_root / ARCHIVE.name
        DRIVE_SIDECAR = drive_root / SIDECAR.name
        _require_absent([DRIVE_ARCHIVE, DRIVE_SIDECAR])
    argv = [sys.executable, str(WORK/RUNNER_PATH), '--generate', '--diagnostic-bootstrap', str(DIAGNOSTIC_BOOTSTRAP), '--output', str(OUTPUT), '--source-commit', AUTHORIZED_REF]
    _write_bytes_exclusive(LOG/'command.json', (json.dumps(argv)+'\n').encode())
    runner_invocation_attempted = True
    state['runner_invocation_attempted'] = True
    completed = subprocess.run(argv, cwd=WORK, capture_output=True, text=True)
    runner_started = True
    state.update({'runner_started': True, 'runner_returned': True})
    _write_bytes_exclusive(LOG/'runner.stdout', completed.stdout.encode())
    _write_bytes_exclusive(LOG/'runner.stderr', completed.stderr.encode())
    response_lines = [line for line in completed.stdout.splitlines() if line.strip()]
    if len(response_lines) != 1:
        raise RuntimeError('runner did not emit exactly one stdout response line')
    response = json.loads(response_lines[0])
    actual_package_path = _validated_actual_package(response['actual_package_path'], OUTPUT)
    state['actual_package_path'] = str(actual_package_path)
    audit = json.loads((actual_package_path/'audit.json').read_text())
    expected_exit = {'RC0_P_PASS_R_PASS': 0, 'RC0_P_FAIL': 3, 'RC0_P_PASS_R_FAIL': 3, 'INSUFFICIENT_EVIDENCE': 4, 'INVALID_EXPERIMENT': 2}
    if audit.get('status') not in expected_exit or completed.returncode != expected_exit[audit['status']]:
        raise RuntimeError('runner exit/status mismatch')
    diagnostic_values = {'FEASIBLE', 'NOT_FEASIBLE', 'INSUFFICIENT_TO_DECIDE'}
    if audit.get('diagnostic_class') != 'DIAGNOSTIC_ONLY' or audit.get('step1_carrier_survival') not in diagnostic_values or audit.get('step2_relation_readout') not in diagnostic_values:
        raise RuntimeError('runner diagnostic question schema mismatch')
    state.update({'audit_loaded': True, 'audit_status': audit['status']})
except Exception as exc:
    caught = exc
    state['failure_type'] = type(exc).__name__
finally:
    state.update({'runner_invocation_attempted': runner_invocation_attempted, 'runner_started': runner_started})
    try:
        if RUN_ID is None:
            raise RuntimeError('diagnostic run identity unavailable; no bound archive path can be created')
        _require_absent([ARCHIVE_ROOT, ARCHIVE, SIDECAR])
        ARCHIVE_ROOT.mkdir(mode=0o700, parents=False, exist_ok=False)
        _write_bytes_exclusive(ARCHIVE_ROOT/'execution_state.json', (json.dumps(state, sort_keys=True, indent=2)+'\n').encode())
        if log_owned:
            _copy_tree_exclusive(LOG, ARCHIVE_ROOT/'notebook-log')
        if actual_package_path is not None:
            safe_actual = _validated_actual_package(str(actual_package_path), OUTPUT)
            _copy_tree_exclusive(safe_actual, ARCHIVE_ROOT/'actual-package')
        _zip_tree_exclusive(ARCHIVE_ROOT, ARCHIVE)
        with zipfile.ZipFile(ARCHIVE) as handle:
            if handle.testzip() is not None:
                raise RuntimeError('local archive readability check failed')
        archive_bytes = ARCHIVE.read_bytes()
        archive_sha = hashlib.sha256(archive_bytes).hexdigest()
        sidecar_payload = (json.dumps({'schema': 'sc-sstw.rc0.archive-sidecar.v1', 'diagnostic_class': 'DIAGNOSTIC_ONLY', 'run_id': RUN_ID, 'source_ref': AUTHORIZED_REF, 'diagnostic_bootstrap_sha256': diagnostic_bootstrap_sha, 'archive_name': ARCHIVE.name, 'archive_size': len(archive_bytes), 'archive_sha256': archive_sha, 'formal_result': False, 'stage_progression_allowed': False}, sort_keys=True, indent=2)+'\n').encode()
        _write_bytes_exclusive(SIDECAR, sidecar_payload)
        if AUTHORIZE_DRIVE_IO:
            _copy_file_exclusive(ARCHIVE, DRIVE_ARCHIVE)
            _copy_file_exclusive(SIDECAR, DRIVE_SIDECAR)
            for local_path, drive_path in ((ARCHIVE, DRIVE_ARCHIVE), (SIDECAR, DRIVE_SIDECAR)):
                local_bytes = local_path.read_bytes()
                drive_bytes = drive_path.read_bytes()
                if drive_path.stat().st_size != local_path.stat().st_size or hashlib.sha256(drive_bytes).hexdigest() != hashlib.sha256(local_bytes).hexdigest():
                    raise RuntimeError(f'Drive readback identity mismatch: {drive_path.name}')
            if DRIVE_SIDECAR.read_bytes() != sidecar_payload:
                raise RuntimeError('Drive sidecar content mismatch')
            with zipfile.ZipFile(DRIVE_ARCHIVE) as handle:
                if handle.testzip() is not None:
                    raise RuntimeError('Drive ZIP readback is corrupt')
    except Exception as exc:
        archive_error = exc
        print(json.dumps({'status': 'INVALID_EXPERIMENT', 'reason_code': 'NOTEBOOK_ARCHIVE_FAILURE', 'runner_started': runner_started, 'archive_error_type': type(exc).__name__}, sort_keys=True))
if caught is not None:
    raise caught
if archive_error is not None:
    raise archive_error
print(json.dumps({'diagnostic_class': audit['diagnostic_class'], 'step1_carrier_survival': audit['step1_carrier_survival'], 'step2_relation_readout': audit['step2_relation_readout'], 'route_decision': audit['route_decision'], 'exit_code': completed.returncode, 'status': audit['status'], 'runner_started': runner_started, 'archive_sha256': archive_sha, 'sidecar': str(SIDECAR)}, sort_keys=True))
